# Projeto Integrador — Análise Exploratória (UCI Credit Card)

Nomes: Lucas Souza Santos, Ali Ahmad, Leonardo Tonon

Notebook limpo e reorganizado em português. Responde às questões: escolha do tema/base, compreensão dos dados, estatísticas, visualizações, outliers, segmentações, tendências temporais e aplicações.

## 1) Escolha do Tema
Análise exploratória para identificar fatores associados à inadimplência de cartões de crédito e gerar insights para gestão de risco.

## 2) Escolha da Base
Base: UCI Credit Card (arquivo esperado: `/content/UCI_Credit_Card.csv` ou coloque o CSV na raiz do ambiente). Contém dados demográficos, limites, histórico de faturas e pagamentos e a variável alvo `default.payment.next.month`.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-darkgrid')

csv_path = '/content/UCI_Credit_Card.csv'
try:
    df = pd.read_csv(csv_path)
    print(f'Dataset carregado: {csv_path}')
except Exception as e:
    print('Erro ao carregar CSV:', e)
    try:
        df = pd.read_csv('UCI_Credit_Card.csv')
        print('Dataset carregado: UCI_Credit_Card.csv (raiz do workspace)')
    except Exception as e2:
        print('Falha ao localizar o arquivo. Coloque o CSV em /content/ ou na raiz do workspace.')
        raise

df.head()

## 3) Compreensão dos dados
Respostas automáticas e verificações: estrutura, tipos, nulos e duplicados.

In [ ]:
print('Shape (linhas, colunas):', df.shape)
print('\nTipos de dados:')
print(df.dtypes)

print('\nValores ausentes por coluna:')
print(df.isnull().sum())

print('\nLinhas duplicadas (total):', df.duplicated().sum())

# Verificar colunas categóricas esperadas
for col in ['SEX','EDUCATION','MARRIAGE']:
    if col in df.columns:
        print(f"\nContagens - {col}:\n", df[col].value_counts(dropna=False))

# Criar faixa etária
if 'AGE' in df.columns:
    bins = [0,20,30,40,50,60,70,120]
    labels = ['<20','20-29','30-39','40-49','50-59','60-69','70+']
    df['AGE_GROUP'] = pd.cut(df['AGE'], bins=bins, labels=labels, right=False)
    print('\nContagem por AGE_GROUP:')
    print(df['AGE_GROUP'].value_counts())

## 3c/d Resumo curto (comportamento detectado)
- Estrutura: linhas x colunas acima.
- Tipos: mistura de numéricas contínuas e variáveis categóricas codificadas numericamente.
- Valores ausentes: verificado por isnull().
- Duplicados: verificado e exibido.

## 4) Estatísticas Descritivas
Calcular médias, medianas, min, max e desvio padrão para variáveis numéricas relevantes.

In [ ]:
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print('Colunas numéricas:', len(num_cols))
describe = df[num_cols].describe().T
# adicionar mediana
describe['median'] = df[num_cols].median()
# selecionar colunas de interesse para exibir
display_cols = ['count','mean','median','std','min','25%','50%','75%','max']
display(describe[display_cols])

## 5) Visualização e Padrões
5a) Correlação entre variáveis numéricas. 5b) Histogramas e boxplots para LIMIT_BAL, AGE, BILL_AMT1..6, PAY_AMT1..6.

In [ ]:
corr = df.select_dtypes(include=['number']).corr()
plt.figure(figsize=(12,10))
sns.heatmap(corr, cmap='coolwarm', center=0, fmt='.2f')
plt.title('Heatmap de correlação')
plt.show()

In [ ]:
# Histogramas e boxplots (selecionadas)
cols_hist = ['LIMIT_BAL','AGE'] + [f'BILL_AMT{i}' for i in range(1,7) if f'BILL_AMT{i}' in df.columns]
cols_hist = [c for c in cols_hist if c in df.columns]

plt.figure(figsize=(14,8))
for i, col in enumerate(cols_hist,1):
    plt.subplot(2, 4, i)
    sns.histplot(df[col].dropna(), kde=True)
    plt.title(col)
plt.tight_layout()
plt.show()

plt.figure(figsize=(14,8))
for i, col in enumerate(cols_hist,1):
    plt.subplot(2, 4, i)
    sns.boxplot(x=df[col].dropna())
    plt.title(col)
plt.tight_layout()
plt.show()

## 6) Outliers e Anomalias
6a) Identificação por IQR; 6b) anomalias em categorias (EDUCATION, MARRIAGE).

In [ ]:
# Contagem de outliers por IQR para colunas financeiras
fin_cols = [c for c in ['LIMIT_BAL'] + [f'BILL_AMT{i}' for i in range(1,7)] + [f'PAY_AMT{i}' for i in range(1,7)] if c in df.columns]
outliers = {}
for c in fin_cols:
    Q1 = df[c].quantile(0.25)
    Q3 = df[c].quantile(0.75)
    IQR = Q3 - Q1
    low = Q1 - 1.5 * IQR
    high = Q3 + 1.5 * IQR
    out_count = df[(df[c] < low) | (df[c] > high)].shape[0]
    outliers[c] = out_count
print('Outliers (IQR) por coluna:')
for k,v in outliers.items():
    print(f'{k}: {v} ({v/len(df):.2%})')

# Anomalias categóricas
for c in ['EDUCATION','MARRIAGE']:
    if c in df.columns:
        print(f"\nCategorias em {c}:\n", df[c].value_counts())

## 7) Comparações e Segmentações
7a) Taxa de inadimplência por sexo, escolaridade, estado civil e faixa etária.
7b) Comportamento ao longo do tempo (médias de PAY_X, BILL_AMT, PAY_AMT).
7c) Insights para decisão.

In [ ]:
# Verificar coluna alvo
alvo = 'default.payment.next.month'
if alvo not in df.columns:
    print('Coluna alvo não encontrada:', alvo)
else:
    def fmt_rate(s):
        return s.mean().apply(lambda x: f'{x:.2%}') if hasattr(s, 'mean') else ''

    for c in ['SEX','EDUCATION','MARRIAGE','AGE_GROUP']:
        if c in df.columns:
            print(f"\nTaxa de inadimplência por {c}:")
            print(df.groupby(c)[alvo].mean().apply(lambda x: f'{x:.2%}'))

# Tendências temporais
pay_cols = [col for col in df.columns if col.startswith('PAY_')]
bill_cols = [col for col in df.columns if col.startswith('BILL_AMT')]
pay_amt_cols = [col for col in df.columns if col.startswith('PAY_AMT')]

pay_means = df[pay_cols].mean() if pay_cols else pd.Series()
bill_means = df[bill_cols].mean() if bill_cols else pd.Series()
pay_amt_means = df[pay_amt_cols].mean() if pay_amt_cols else pd.Series()

# rótulos de meses (apenas ilustrativos: BILL_AMT1 = Set etc.)
months = ['Set','Ago','Jul','Jun','Mai','Abr']

if not pay_means.empty:
    plt.figure(figsize=(8,4))
    sns.lineplot(x=range(len(pay_means)), y=pay_means.values, marker='o')
    plt.xticks(range(len(pay_means)), pay_means.index, rotation=45)
    plt.title('Média PAY_X (tempo)')
    plt.tight_layout(); plt.show()

if not bill_means.empty:
    plt.figure(figsize=(8,4))
    sns.lineplot(x=range(len(bill_means)), y=bill_means.values, marker='o')
    plt.xticks(range(len(bill_means)), bill_means.index, rotation=45)
    plt.title('Média BILL_AMT (tempo)')
    plt.tight_layout(); plt.show()

if not pay_amt_means.empty:
    plt.figure(figsize=(8,4))
    sns.lineplot(x=range(len(pay_amt_means)), y=pay_amt_means.values, marker='o')
    plt.xticks(range(len(pay_amt_means)), pay_amt_means.index, rotation=45)
    plt.title('Média PAY_AMT (tempo)')
    plt.tight_layout(); plt.show()

## 8) Contextualização e Aplicação
Resuma e aplique os insights: ações de negócio e avaliação da pertinência da base para o problema.

### Resumo executivo (ações recomendadas)
- Agrupar categorias ruidosas: EDUCATION (0,4,5,6) → 'Outros'; MARRIAGE (0) → 'Desconhecido/Outros'.
- Tratar outliers em variáveis financeiras (capping ou transformação log).
- Criar features: taxa de utilização do limite, média de atrasos, variação mensal da fatura.
- Construir modelos de scoring usando histórico PAY_X e BILL_AMT como principais preditores.

## Q&A — Respostas diretas às perguntas solicitadas
1) Escolha do Tema: Predição/entendimento de inadimplência em cartão de crédito.
2) Escolha da Base: UCI Credit Card (histórico de 6 meses + demográficos + alvo `default.payment.next.month`).
3a) Estrutura: ver célula de `df.shape` (ex.: 30000 x 25). 3b) Tipos: numéricas contínuas (LIMIT_BAL, BILL_AMT*, PAY_AMT*), categóricas codificadas (SEX, EDUCATION, MARRIAGE, PAY_X). 3c) Valores ausentes: ver `df.isnull().sum()`; nesta base original não há nulos. 3d) Duplicados/inconsistências: ver `df.duplicated()` e `value_counts()`; anomalias em EDUCATION/MARRIAGE detectadas.
4a) Estatísticas básicas: ver `df.describe()`; média, mediana, min, max e std exibidos na seção 4.
5a) Relações: heatmap mostra correlações fortes entre BILL_AMT consecutivos; 5b) Visualizações: histogramas, boxplots e heatmap implementados.
6a) Outliers: contagem por IQR mostrada; 6b) Anomalias: categorias 0,4,5,6 em EDUCATION e 0 em MARRIAGE precisariam recodificação.
7a) Diferenças por grupo: taxas de inadimplência por SEX/EDUCATION/MARRIAGE/AGE_GROUP exibidas; 7b) Temporal: médias PAY_X, BILL_AMT, PAY_AMT plotadas; 7c) Insights: segmentação de risco, ajuste de políticas de crédito e réguas de cobrança.
8a) Aplicação: modelos de score, políticas de crédito segmentadas, campanhas de marketing e estratégias de cobrança personalizadas. 8b) Pertinência: sim — base adequada para modelagem de inadimplência.

---

Observação: revise caminhos de arquivo antes de executar; salve backups se desejar manter o notebook original.